# In-Class Competition: Home Credit Default Risk
## Complete Ablation-Controlled Ensemble Pipeline (LightGBM + CatBoost + XGBoost)

Targeting ROC-AUC with strictly legitimate methods (no external data, leak-free CV).

### Baseline to beat:
- **Baseline OOF ROC-AUC: 0.75849** (Public LB: 0.76425)

### Ablation Toggles included:
1. `TOGGLE_PARAM_FIXES`: `subsample_freq=1`, `num_leaves=24`, `min_child_samples=65`
2. `TOGGLE_EXTRA_FEATURES`: `MISSING_COUNT`, `EXT_SOURCES_NAN_COUNT`, markup, spreads
3. `TOGGLE_TARGET_ENCODE`: Leak-free smoothed Out-of-Fold target encoding for high-cardinality nominals
4. `TOGGLE_CATBOOST`: Train CatBoost on identical 5 folds
5. `TOGGLE_XGBOOST`: Train XGBoost on identical 5 folds


### 1. Environment & Dataset Auto-Detection


In [ ]:
import sys
import os
import glob
from pathlib import Path

# Auto-locate dataset on Kaggle, Colab, or local PC
if os.path.exists('/kaggle'):
    train_files = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if train_files:
        source_dir = os.path.dirname(train_files[0])
        os.system(f'rm -rf ./input && ln -s "{source_dir}" ./input')
        print(f'[Kaggle] Linked dataset: {source_dir} -> ./input')
    else:
        print('[Kaggle] Please attach the dataset via the right sidebar.')
elif 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd '/content/drive/MyDrive/competition'
else:
    print('[Local] Running in local environment.')

!ls -lh input


### 2. Imports & Settings


In [ ]:
import gc
import time
import zipfile
import warnings
from scipy.stats import rankdata
from scipy.optimize import minimize

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
print(f"NumPy: {np.__version__}, Pandas: {pd.__version__}, LightGBM: {lgb.__version__}")


### 3. Ablation Toggles
Toggle improvements to test each change against your 0.75849 baseline.


In [ ]:
# ==============================================================================
# EXPERIMENT ABLATION TOGGLES
# ==============================================================================
TOGGLE_PARAM_FIXES    = True   # [1] subsample_freq=1, num_leaves=24, min_child=65
TOGGLE_EXTRA_FEATURES = True   # [2] MISSING_COUNT, EXT_NAN_COUNT, markup, spreads
TOGGLE_TARGET_ENCODE  = False  # [3] Leak-free smoothed OOF target encoding (set True to test)
TOGGLE_CATBOOST       = True   # [4] Train CatBoost on identical 5 folds
TOGGLE_XGBOOST        = True   # [5] Train XGBoost on identical 5 folds
SEEDS                 = [42]   # Set to [42, 2024] for multi-seed averaging
# ==============================================================================
BASELINE_OOF = 0.75849


### 4. Data Loading


In [ ]:
current_dir = Path.cwd()
input_dir = current_dir / 'input'

train = pd.read_csv(input_dir / 'train.csv')
test = pd.read_csv(input_dir / 'test.csv')
sample_sub = pd.read_csv(input_dir / 'sample_submission.csv')

y = train['TARGET'].values
test_ids = test['SK_ID_CURR']

print(f"Train shape: {train.shape}, Test shape: {test.shape}")
print(f"Target default rate: {y.mean():.4%}")


### 5. Feature Engineering (Original 83 Features + High-Value Additions)


In [ ]:
def build_features(df):
    df = df.copy()
    
    # Anomaly indicator & replacement
    df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    
    # Core domain ratios
    df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
    df['INCOME_CREDIT_DIFF'] = df['AMT_INCOME_TOTAL'] - df['AMT_CREDIT']
    df['GOODS_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + 1)
    df['GOODS_CREDIT_DIFF'] = df['AMT_GOODS_PRICE'] - df['AMT_CREDIT']
    
    # Per-person family metrics
    fam_members = df['CNT_FAM_MEMBERS'].fillna(1).replace(0, 1)
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / fam_members
    df['CREDIT_PER_PERSON'] = df['AMT_CREDIT'] / fam_members
    df['ANNUITY_PER_PERSON'] = df['AMT_ANNUITY'] / fam_members
    
    # Time and age ratios
    df['DAYS_EMPLOYED_PERC'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365.25
    df['EMPLOYED_YEARS'] = -df['DAYS_EMPLOYED'] / 365.25
    df['REGISTRATION_BIRTH_RATIO'] = df['DAYS_REGISTRATION'] / df['DAYS_BIRTH']
    df['ID_PUBLISH_BIRTH_RATIO'] = df['DAYS_ID_PUBLISH'] / df['DAYS_BIRTH']
    df['PHONE_BIRTH_RATIO'] = df['DAYS_LAST_PHONE_CHANGE'] / df['DAYS_BIRTH']
    df['PHONE_EMPLOYED_RATIO'] = df['DAYS_LAST_PHONE_CHANGE'] / (df['DAYS_EMPLOYED'] - 1)
    
    # Car features
    df['FLAG_OWN_CAR'] = df['OWN_CAR_AGE'].notnull().astype(int)
    df['CAR_EMPLOYED_RATIO'] = (df['OWN_CAR_AGE'] * 365) / (-df['DAYS_EMPLOYED'] + 1)
    df['CAR_BIRTH_RATIO'] = (df['OWN_CAR_AGE'] * 365) / (-df['DAYS_BIRTH'] + 1)
    
    # EXT_SOURCE combinations
    ext_sources = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    df['EXT_SOURCES_MEAN'] = df[ext_sources].mean(axis=1)
    df['EXT_SOURCES_STD'] = df[ext_sources].std(axis=1)
    df['EXT_SOURCES_MIN'] = df[ext_sources].min(axis=1)
    df['EXT_SOURCES_MAX'] = df[ext_sources].max(axis=1)
    df['EXT_SOURCES_PROD'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
    df['EXT_1_x_EXT_2'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2']
    df['EXT_2_x_EXT_3'] = df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
    df['EXT_1_x_EXT_3'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_3']
    df['EXT_1_over_BIRTH'] = df['EXT_SOURCE_1'] / df['DAYS_BIRTH']
    df['EXT_2_over_BIRTH'] = df['EXT_SOURCE_2'] / df['DAYS_BIRTH']
    df['EXT_3_over_BIRTH'] = df['EXT_SOURCE_3'] / df['DAYS_BIRTH']
    df['EXT_2_over_CREDIT'] = df['EXT_SOURCE_2'] / (df['AMT_CREDIT'] + 1)
    df['EXT_3_over_CREDIT'] = df['EXT_SOURCE_3'] / (df['AMT_CREDIT'] + 1)
    
    # Contacts & Bureau
    df['FLAG_CONTACTS_SUM'] = df['FLAG_MOBIL'] + df['FLAG_EMP_PHONE'] + df['FLAG_WORK_PHONE']
    bureau_cols = ['AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_MON', 
                   'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR']
    df['BUREAU_ENQUIRIES_SUM'] = df[bureau_cols].sum(axis=1)
    
    # Extra High-Value Features (Controlled by Toggle)
    if TOGGLE_EXTRA_FEATURES:
        df['MISSING_COUNT'] = df.isnull().sum(axis=1)
        df['EXT_SOURCES_NAN_COUNT'] = df[ext_sources].isnull().sum(axis=1)
        df['CREDIT_TO_GOODS_MARKUP'] = (df['AMT_CREDIT'] - df['AMT_GOODS_PRICE']) / (df['AMT_GOODS_PRICE'] + 1)
        df['ANNUITY_MONTHLY_INCOME_RATIO'] = df['AMT_ANNUITY'] / ((df['AMT_INCOME_TOTAL'] / 12) + 1)
        df['EXT_2_minus_EXT_3'] = df['EXT_SOURCE_2'] - df['EXT_SOURCE_3']
        df['EXT_1_minus_EXT_2'] = df['EXT_SOURCE_1'] - df['EXT_SOURCE_2']
        df['BUREAU_RECENT_RATIO'] = (df['AMT_REQ_CREDIT_BUREAU_HOUR'] + df['AMT_REQ_CREDIT_BUREAU_MON']) / (df['BUREAU_ENQUIRIES_SUM'] + 1)
    
    return df

print("Building features...")
train_fe = build_features(train)
test_fe = build_features(test)

# Unsupervised Group Statistics & Frequencies across combined train+test
combined = pd.concat([train_fe, test_fe], axis=0, ignore_index=True)
for col in ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE']:
    freq = combined[col].value_counts(normalize=True)
    train_fe[f'{col}_FREQ'] = train_fe[col].map(freq)
    test_fe[f'{col}_FREQ'] = test_fe[col].map(freq)

for grp_col in ['NAME_EDUCATION_TYPE', 'OCCUPATION_TYPE']:
    for num_col in ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'EXT_SOURCE_2']:
        grp_stats = combined.groupby(grp_col)[num_col].agg(['mean']).reset_index()
        grp_stats.columns = [grp_col, f'{grp_col}_{num_col}_MEAN']
        train_fe = train_fe.merge(grp_stats, on=grp_col, how='left')
        test_fe = test_fe.merge(grp_stats, on=grp_col, how='left')
        train_fe[f'{num_col}_RATIO_{grp_col}'] = train_fe[num_col] / (train_fe[f'{grp_col}_{num_col}_MEAN'] + 1)
        test_fe[f'{num_col}_RATIO_{grp_col}'] = test_fe[num_col] / (test_fe[f'{grp_col}_{num_col}_MEAN'] + 1)

print(f"Total features engineered: {train_fe.shape[1] - 2}")


### 6. Categorical Encoding


In [ ]:
ignore_cols = ['SK_ID_CURR', 'TARGET']
features = [c for c in train_fe.columns if c not in ignore_cols]
cat_cols = train_fe[features].select_dtypes(include=['object', 'category']).columns.tolist()

for c in cat_cols:
    le = LabelEncoder()
    unique_vals = list(train_fe[c].astype(str).unique()) + list(test_fe[c].astype(str).unique())
    le.fit(unique_vals)
    train_fe[c] = pd.Categorical(le.transform(train_fe[c].astype(str)))
    test_fe[c] = pd.Categorical(le.transform(test_fe[c].astype(str)))

X = train_fe[features].copy()
X_test = test_fe[features].copy()
print(f"Data ready: {X.shape[1]} features, {len(X)} rows")


### 7. Model Training (Identical 5-Fold Stratified CV)


In [ ]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

if TOGGLE_PARAM_FIXES:
    print('>> Using Regularized LightGBM (subsample_freq=1, min_child=65, num_leaves=24)')
    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': 0.02,
        'num_leaves': 24,
        'max_depth': -1,
        'min_child_samples': 65,
        'subsample': 0.8,
        'subsample_freq': 1,
        'colsample_bytree': 0.65,
        'reg_alpha': 0.5,
        'reg_lambda': 10.0,
        'n_estimators': 2500,
        'verbose': -1,
        'n_jobs': -1
    }
else:
    print('>> Using Original LightGBM baseline parameters')
    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': 0.03,
        'num_leaves': 31,
        'max_depth': -1,
        'min_child_samples': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.7,
        'reg_alpha': 0.1,
        'reg_lambda': 10.0,
        'n_estimators': 1500,
        'verbose': -1,
        'n_jobs': -1
    }

oof_lgb = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))

oof_cb = np.zeros(len(X))
test_cb = np.zeros(len(X_test))

oof_xgb = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))

feature_importances = pd.DataFrame(index=features)

# Numeric representations for CatBoost and XGBoost
X_num = X.copy()
X_test_num = X_test.copy()
for c in cat_cols:
    X_num[c] = X_num[c].astype('category').cat.codes
    X_test_num[c] = X_test_num[c].astype('category').cat.codes

for s_idx, seed in enumerate(SEEDS):
    print(f"\n==================== SEED {seed} ({s_idx + 1}/{len(SEEDS)}) ====================")
    lgb_p = lgb_params.copy()
    lgb_p['random_state'] = seed
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y[train_idx]
        X_va, y_va = X.iloc[val_idx], y[val_idx]
        
        # --- A. LIGHTGBM ---
        model_lgb = lgb.LGBMClassifier(**lgb_p)
        model_lgb.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_names=['valid'],
            categorical_feature=cat_cols,
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )
        pred_val_lgb = model_lgb.predict_proba(X_va)[:, 1]
        oof_lgb[val_idx] += pred_val_lgb / len(SEEDS)
        test_lgb += model_lgb.predict_proba(X_test)[:, 1] / (n_splits * len(SEEDS))
        feature_importances[f'fold_{fold + 1}_s{seed}'] = model_lgb.feature_importances_
        
        # --- B. CATBOOST ---
        if TOGGLE_CATBOOST:
            cb_model = CatBoostClassifier(
                iterations=1500, learning_rate=0.03, depth=6, eval_metric='AUC',
                random_seed=seed + fold, early_stopping_rounds=100, verbose=False,
                task_type='CPU'
            )
            cb_model.fit(
                X_num.iloc[train_idx], y_tr,
                eval_set=(X_num.iloc[val_idx], y_va)
            )
            oof_cb[val_idx] += cb_model.predict_proba(X_num.iloc[val_idx])[:, 1] / len(SEEDS)
            test_cb += cb_model.predict_proba(X_test_num)[:, 1] / (n_splits * len(SEEDS))
            
        # --- C. XGBOOST ---
        if TOGGLE_XGBOOST:
            xgb_model = XGBClassifier(
                n_estimators=1500, learning_rate=0.03, max_depth=5, subsample=0.8,
                colsample_bytree=0.65, eval_metric='auc', random_state=seed + fold,
                early_stopping_rounds=100, tree_method='hist'
            )
            xgb_model.fit(
                X_num.iloc[train_idx], y_tr,
                eval_set=[(X_num.iloc[val_idx], y_va)], verbose=False
            )
            oof_xgb[val_idx] += xgb_model.predict_proba(X_num.iloc[val_idx])[:, 1] / len(SEEDS)
            test_xgb += xgb_model.predict_proba(X_test_num)[:, 1] / (n_splits * len(SEEDS))
            
        print(f"Fold {fold + 1} LGBM AUC: {roc_auc_score(y_va, pred_val_lgb):.5f}")


### 8. Ablation Comparison against 0.75849 Baseline & Rank-Average Blending


In [ ]:
auc_lgb = roc_auc_score(y, oof_lgb)
print("\n" + "=" * 60)
print(f"BASELINE OOF ROC-AUC        : {BASELINE_OOF:.5f}")
print(f"CURRENT LIGHTGBM OOF ROC-AUC : {auc_lgb:.5f} (Delta: {auc_lgb - BASELINE_OOF:+.5f})")

final_test_preds = test_lgb

active_models_oof = [('LightGBM', oof_lgb, test_lgb)]
if TOGGLE_CATBOOST:
    auc_cb = roc_auc_score(y, oof_cb)
    print(f"CATBOOST OOF ROC-AUC        : {auc_cb:.5f} (Delta: {auc_cb - BASELINE_OOF:+.5f})")
    active_models_oof.append(('CatBoost', oof_cb, test_cb))
if TOGGLE_XGBOOST:
    auc_xgb = roc_auc_score(y, oof_xgb)
    print(f"XGBOOST OOF ROC-AUC         : {auc_xgb:.5f} (Delta: {auc_xgb - BASELINE_OOF:+.5f})")
    active_models_oof.append(('XGBoost', oof_xgb, test_xgb))

if len(active_models_oof) > 1:
    print("\n--- Optimizing Rank-Average Blend Weights on OOF Predictions ---")
    K = len(active_models_oof)
    ranked_oofs = [rankdata(m[1]) / len(y) for m in active_models_oof]
    ranked_tests = [rankdata(m[2]) / len(test_ids) for m in active_models_oof]
    
    def blend_objective(weights):
        w = np.array(weights)
        w = w / np.sum(w)
        blended = sum(w[i] * ranked_oofs[i] for i in range(K))
        return -roc_auc_score(y, blended)
    
    init_weights = [1.0 / K] * K
    bounds = [(0, 1)] * K
    res = minimize(blend_objective, init_weights, bounds=bounds, method='SLSQP')
    opt_w = res.x / np.sum(res.x)
    
    for i, m in enumerate(active_models_oof):
        print(f"  > {m[0]} Optimal Weight: {opt_w[i]:.3f}")
        
    best_ensemble_auc = -res.fun
    print(f"OPTIMAL ENSEMBLE OOF ROC-AUC: {best_ensemble_auc:.5f} (Total Gain over Baseline: {best_ensemble_auc - BASELINE_OOF:+.5f})")
    print("=" * 60)
    
    final_test_preds = sum(opt_w[i] * ranked_tests[i] for i in range(K))


### 9. Feature Importance Analysis


In [ ]:
feature_importances['mean_importance'] = feature_importances.mean(axis=1)
top25 = feature_importances['mean_importance'].sort_values(ascending=False).head(25)

plt.figure(figsize=(10, 8))
sns.barplot(x=top25.values, y=top25.index)
plt.title('Top 25 Features by Importance')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


### 10. Save Submission & Create Reproducible Honors Archive


In [ ]:
output_dir = current_dir / 'output'
os.makedirs(output_dir, exist_ok=True)
sub_file = output_dir / 'submission.csv'

submission = pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET': final_test_preds
})
submission.to_csv(sub_file, index=False)
print(f"Saved final submission file to: {sub_file}")
print(f"Submission shape: {submission.shape}")
print(submission.head())

# Create zip archive for Honors requirement
nb_name = 'champion_solution.ipynb'
zip_file = current_dir / 'champion_solution.zip'

nb_file = current_dir / nb_name
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    if nb_file.exists():
        zf.write(nb_file, arcname=nb_name)

print(f"Created Honors zip file: {zip_file}")


### 11. Download Outputs


In [ ]:
from IPython.display import FileLink, display

if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(sub_file))
    files.download(str(zip_file))
else:
    print('Click the links below to download your files:')
    display(FileLink('output/submission.csv'))
    display(FileLink('champion_solution.zip'))
